### Load & clean:

In [1]:
import pandas as pd 
import numpy as np 

df = pd.read_csv('data/telco_churn.csv')

#Fix Total Charges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

#Drop CoustmerId
df.drop('customerID', axis=1, inplace=True)

#Encode Target
df['Churn'].map({'Yes' : 1 , 'No' : 0 })

print("Shape after cleaning", df.shape)
print("Churn Counts", df['Churn'].value_counts().to_dict())

Shape after cleaning (7043, 20)
Churn Counts {'No': 5174, 'Yes': 1869}


C:\Users\dell\AppData\Local\Temp\ipykernel_4508\3116358995.py:8: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


### Feature Engineering + Test Train Spilt

In [8]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

#Seprate Features
categorical_cols = df.select_dtypes(include='object').columns.tolist()
numerical_columns = ['Tenure', 'MonthlyCharges', 'TotalCharges']

#Label encode
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

print("All Columns Encoded")
df.head()

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

imputer = SimpleImputer(strategy='mean')

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

scaled = StandardScaler()
X_train_scaled = scaled.fit_transform(X_train)
X_test_scaled = scaled.transform(X_test)

print("Shape of X Train", X_train.shape)
print("Shape of X Test", X_test.shape)
print("Features", X.columns.tolist())

All Columns Encoded
Shape of X Train (5282, 19)
Shape of X Test (1761, 19)
Features ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


###  Train model:

In [9]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

models = {
    "Logistic Regression" : LogisticRegression(max_iter=1000),
    "Random Forest" : RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boost" : GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:,1])
    results[name] = auc
    print(f"\n{name}")
    print(classification_report(y_test, y_pred))
    print(f"AUC Score: {auc:.4f}")
best_model_name = max(results, key=results.get)
print(f"\n🏆 Best model: {best_model_name} (AUC: {results[best_model_name]:.4f})")


Logistic Regression
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1294
           1       0.65      0.56      0.60       467

    accuracy                           0.80      1761
   macro avg       0.75      0.72      0.73      1761
weighted avg       0.80      0.80      0.80      1761

AUC Score: 0.8440

Random Forest
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1294
           1       0.64      0.48      0.55       467

    accuracy                           0.79      1761
   macro avg       0.74      0.69      0.71      1761
weighted avg       0.78      0.79      0.78      1761

AUC Score: 0.8270

Gradient Boost
              precision    recall  f1-score   support

           0       0.83      0.92      0.87      1294
           1       0.67      0.48      0.56       467

    accuracy                           0.80      1761
   macro avg       0.75      0.70      0

### Save the best model


In [13]:
import joblib, os

best_model = models[best_model_name]

os.makedirs('../app/model', exist_ok=True)

joblib.dump(best_model, '../app/model/churn_model.pkl')
joblib.dump(scaled, '../app/model/scaled_model.pkl')

#Save Features Names
features_name = X.columns.tolist()
joblib.dump(features_name, '../app/model/features_name.pkl')

print("\nFeatures used", features_name)


Features used ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
